# E09 01 - RAG con LangChain (Resolution)

Este ejercicio introduce RAG y agentes de forma minima.

RAG significa:

```txt
pregunta -> retriever -> contexto -> prompt -> LLM -> respuesta fundamentada
```

La idea central: el modelo no responde solo con memoria interna; primero recibe documentos recuperados.


## Antes de tocar codigo: que estamos construyendo

Este notebook esta pensado para que puedas entenderlo aunque lo abras sin ver la clase.

Tema: **RAG con LangChain**.

La regla didactica es:

1. primero explicamos el concepto;
2. despues mostramos el codigo minimo;
3. despues conectamos ese codigo con el paso anterior;
4. finalmente ejecutamos y leemos el resultado.

Cuando veas una funcion, preguntate:

- que recibe;
- que devuelve;
- que parte del flujo representa;
- si es logica de negocio, orquestacion o instrumentacion.


## Herramientas RAG que aparecen

| Herramienta | Para que sirve |
|---|---|
| `Document` | Guarda texto y metadata |
| `RecursiveCharacterTextSplitter` | Divide documentos en chunks |
| `OpenAIEmbeddings` | Convierte texto en vectores |
| `Chroma` | Vector store para buscar por similitud |
| `retriever` | Interfaz de busqueda semantica |
| `ChatPromptTemplate` | Prompt que combina contexto + consulta |


In [ ]:
# Esta celda instala las dependencias del notebook.
# En Google Colab cada notebook arranca con un entorno limpio, por eso instalamos al inicio.
# En local, si ya instalaste estos paquetes, pip simplemente confirmara que existen.
!pip install -q langchain langchain-openai langchain-chroma chromadb

print('Dependencias instaladas: langchain langchain-openai langchain-chroma chromadb')


In [ ]:
import os
from getpass import getpass

# Nunca escribimos una API key real dentro del notebook.
# getpass permite pegar la key en ejecucion sin que quede visible en la salida.
# os.environ guarda la key solo para esta sesion de Python.
os.environ['OPENAI_API_KEY'] = getpass('OpenAI API Key: ')

print('OpenAI configurado para esta sesion.')


## Imports

Importamos piezas de RAG, LLM y, si corresponde, LangGraph/Langfuse.


In [ ]:
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


## Documentos

En un proyecto real vendrian de archivos. Aca los dejamos en el notebook para que sea autocontenido.


In [ ]:
from langchain_core.documents import Document

# Document es el contenedor estandar de LangChain para texto + metadata.
# page_content guarda el texto que se va a recuperar.
# metadata guarda informacion util como dominio y fuente.
docs = [
    Document(page_content='RRHH: las vacaciones se piden con 15 dias de anticipacion y aprobacion del manager.', metadata={'domain': 'hr', 'source': 'hr_policy'}),
    Document(page_content='RRHH: la licencia por estudio requiere constancia de examen y registro en PeopleOps.', metadata={'domain': 'hr', 'source': 'hr_policy'}),
    Document(page_content='Tech: para problemas de VPN, reiniciar cliente, validar 2FA y probar otra red.', metadata={'domain': 'tech', 'source': 'tech_guide'}),
    Document(page_content='Tech: el reset de contrasena se realiza desde el portal de identidad corporativa.', metadata={'domain': 'tech', 'source': 'tech_guide'}),
    Document(page_content='Finance: los reembolsos aprobados se procesan los dias 10 y 25 de cada mes.', metadata={'domain': 'finance', 'source': 'finance_policy'}),
    Document(page_content='Finance: las facturas deben cargarse con comprobante, monto y centro de costo.', metadata={'domain': 'finance', 'source': 'finance_policy'}),
]

print('Documentos cargados:', len(docs))


## Chunks y ChromaDB

El splitter crea fragmentos. ChromaDB guarda embeddings y permite recuperar los fragmentos mas parecidos a la pregunta.


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
chunks = splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='m3_examples_rag_agents',
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
print('Chunks:', len(chunks))


## Prompt RAG

El prompt obliga al LLM a usar solo el contexto recuperado.


In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Sos un agente especialista. Responde usando solo el contexto provisto. Si no alcanza, decilo.'),
    ('human', 'Contexto:\n{context}\n\nConsulta:\n{query}\n\nRespuesta:'),
])

def format_docs(retrieved_docs):
    return '\n\n'.join(doc.page_content for doc in retrieved_docs)


## Agente RAG con LangChain puro

Una funcion Python orquesta retrieval y generation.


In [ ]:
rag_chain = rag_prompt | llm | StrOutputParser()

def rag_answer(query: str) -> dict:
    retrieved = retriever.invoke(query)
    context = format_docs(retrieved)
    answer = rag_chain.invoke({'context': context, 'query': query})
    return {
        'query': query,
        'answer': answer,
        'sources': [doc.metadata for doc in retrieved],
    }

print(rag_answer('No puedo conectarme a la VPN'))
